# PenG — AI Học Tập
Clone, install, and test PenG from Google Colab with GPU T4.

**Prerequisites:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Clone repo
!git clone https://github.com/canhcutlo/PenG.git
%cd PenG

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
# 3. Install dependencies
# Install system dependencies (Tesseract OCR)
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie 2>&1 | tail -2

# Pillow>=10.0,<11 for Surya-ocr compatibility on Python 3.12
!pip install -r requirements-colab.txt
!pip install pyngrok nest-asyncio requests

In [ ]:
# 4. Compile check
!python -m compileall app

In [ ]:
# 5. Run unit tests (skip AI model integration tests)
!pytest tests/ -v -m "not integration"

In [ ]:
# 5b. Configure LLM model (edit before starting server)
# Default: Qwen/Qwen2.5-3B-Instruct (~6GB 4-bit, works on T4)
# Smaller/faster: Qwen/Qwen2.5-1.5B-Instruct (~3GB)
# Better quality (needs more VRAM): Qwen/Qwen2.5-7B-Instruct
import os
os.environ['LLM_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'
os.environ['LLM_QUANTIZE'] = 'true'
print('LLM:', os.environ['LLM_MODEL'])

In [ ]:
# 5c. Check GPU memory before loading LLM
import torch
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    free = total - reserved
    print(f'GPU: {total:.1f}GB total, {free:.1f}GB free')
    if free < 7:
        print('WARNING: Less than 7GB free. Consider using Qwen2.5-1.5B or restart runtime.')
else:
    print('No GPU detected. LLM will run on CPU (very slow).')

In [ ]:
# 6. Start FastAPI + ngrok
#    Safe to re-run: kills old uvicorn + ngrok first.
#    NOTE: First LLM load will download ~6GB model weights (3-5 min).
import subprocess, sys, time, requests, os, signal
from pyngrok import ngrok

# Optional: set ngrok token for stable URL (get from https://dashboard.ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# ── Step 1: Kill any existing ngrok tunnels ──
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
        print(f'Disconnected old tunnel: {t.public_url}')
except Exception:
    pass
ngrok.kill()
time.sleep(1)

# ── Step 2: Kill any old uvicorn process on port 8000 ──
import socket
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if is_port_in_use(8000):
    print('Port 8000 is busy — killing old uvicorn...')
    if sys.platform == 'win32':
        os.system('taskkill /F /FI "IMAGENAME eq python.exe" /FI "PID ne {}" >nul 2>&1'.format(os.getpid()))
    else:
        os.system('fuser -k 8000/tcp 2>/dev/null')
    time.sleep(2)

# ── Step 3: Start uvicorn ──
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for uvicorn to boot (up to 15s)
server_ok = False
for _ in range(30):
    time.sleep(0.5)
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=1)
        if r.status_code == 200:
            print("Server started OK")
            server_ok = True
            break
    except requests.RequestException:
        pass

if not server_ok:
    print("ERROR: Server did not start. Check logs or restart runtime.")
else:
    # ── Step 4: Open ngrok tunnel ──
    tunnel = ngrok.connect(8000)
    url = tunnel.public_url

    print(f"\n=== PenG is running ===")
    print(f"Frontend: {url}")
    print(f"API docs: {url}/docs")
    print(f"Health:   {url}/api/health")
    print(f"========================")
print(f"\nIf you need to restart the server, use Runtime -> Restart session.")
print(f"NOTE: First query/quiz will trigger model download (~3-5 min).")

In [ ]:
# 7. Quick end-to-end test (run after server is up)
# This uploads the first PDF found in /content and verifies the pipeline works.
import requests, json, time

base = "http://localhost:8000"

# Health check
r = requests.get(f"{base}/api/health")
print('Health:', r.json())

# Upload a real supported file from /content (edit this path if needed)
from pathlib import Path
test_path = next(iter(Path('/content').glob('*.pdf')), None)
if test_path is None:
    print('No PDF found in /content; skip upload test and use the frontend.')
else:
    with open(test_path, 'rb') as f:
        r = requests.post(f"{base}/api/upload", files={"file": f}, data={"category": "pdf"})
    upload = r.json()
    print('Upload:', upload)

# Poll job until complete (max 30s)
job_id = upload.get('job_id') if test_path else None
if job_id:
    for _ in range(30):
        r = requests.get(f"{base}/api/jobs/{job_id}")
        status = r.json().get('status')
        if status in ('completed', 'failed'):
            print('Job done:', r.json())
            break
        time.sleep(1)
    else:
        print('Job timeout')
else:
    print('No job_id returned')

## Verification checklist

Mở `public_url` trong trình duyệt và kiểm tra từng mục:

- [ ] **Cell 7 pass**: Run cell 7 (Quick test) → upload + job completed
- [ ] **Health**: `/api/health` → `{"status":"ok","db":"ok"}`
- [ ] **Frontend**: `/` → hiện 5 tabs (Upload / Hỏi đáp / Quiz / Mindmap / Lịch sử)
- [ ] **Upload**: Kéo-thả file ảnh/PDF nhỏ → trả `doc_id` + `job_id`; poll `/api/jobs/{job_id}`
- [ ] **Query**: Nhập câu hỏi vào tab Hỏi đáp → trả answer (model download lần đầu ~3-5 phút)
- [ ] **Mindmap**: Nhập `doc_id` vào tab Mindmap → hiển thị markmap
- [ ] **Quiz**: Generate quiz từ `doc_id` → hiển thị câu hỏi → submit → chấm điểm
- [ ] **History**: Tab Lịch sử hiển thị hoạt động đã log

## Troubleshooting

| Vấn đề | Cách sửa |
|---|---|
| Upload file không xử lý (job stuck ở queued) | Kiểm tra `/api/jobs/{job_id}`. Nếu status=failed, xem error_message. Model OCR chưa tải → cần file thực |
| PDF không OCR được | Chạy `from app.services.ocr import ocr_pdf; await ocr_pdf('path')` để test OCR engine |
| ngrok limit 5 tunnels | **Runtime → Restart session** rồi chạy lại từ cell 1. Đừng re-run cell 6 nhiều lần || Frontend stuck ở loading | Mở DevTools (F12) → Network tab → xem API call nào bị lỗi. Có thể server chưa start |
| CUDA out of memory | Chuyển sang Qwen2.5-1.5B (`os.environ['LLM_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'`) hoặc Runtime → Restart session |
| Query trả "Không đủ dữ liệu" | Bình thường nếu tài liệu chưa index xong. Chờ job completed rồi thử lại. |
| Model download chậm | Lần đầu tải Qwen2.5-3B ~6GB từ HuggingFace. Đảm bảo kết nối internet ổn định. |
| Server không khởi động | Runtime → Restart session, chạy lại từ cell 1 |